# ReCompress — reproduce the paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kart-ing/ReCompress/blob/main/notebooks/ReCompress_reproduce.ipynb)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.20786357.svg)](https://doi.org/10.5281/zenodo.20786357)

**Query-aware *rewriting* beats deletion at ~8.5× fewer tokens — a 1.5B model, audited against itself.**

This notebook lets you **reproduce every number and figure in the paper** straight from the
real evaluation results committed in the repo — no GPU, no API keys, no Modal required. Then,
*optionally*, you can run the live query-aware compressor on your own text if you have a
DeepSeek API key.

| Part | Needs | What it does |
|---|---|---|
| **1. Setup** | nothing | pull the real result files from GitHub |
| **2. Headline** | nothing | the 5-bar benchmark table + bar chart |
| **3. Cross-solver audit** | nothing | does the win survive an independent judge? |
| **4. Honesty: mask-the-answer** | nothing | how much of the F1 is the answer span itself |
| **5. Multi-turn crossover** | nothing | total-token curves + the Echidna finding |
| **6. Read real compressions** | nothing | browse the 50 actual compressed examples |
| **7. (Optional) live compression** | a DeepSeek key | run the teacher compressor on your own text |
| **8. Full pipeline** | Modal + GPU | pointers to retrain the distilled model |

> Everything in Parts 1–6 is replayed from `results/*.json` — nothing is synthetic, and it runs
> in vanilla Colab in seconds.

## 1. Setup — fetch the real results (no keys needed)

In [ ]:
import os, json, urllib.request

RAW = "https://raw.githubusercontent.com/Kart-ing/ReCompress/main/results/"
FILES = [
    "cross_solver_audit.json", "mask_symmetric.json", "echidna_ablation_sweep.json",
    "5bar_distilled_hotpotqa.json", "5bar_distilled_2wiki.json",
    "5bar_distilled_musique.json", "5bar_distilled_squad.json",
]
os.makedirs("results", exist_ok=True)
for f in FILES:
    dst = os.path.join("results", f)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(RAW + f, dst)
        print("downloaded", f)
    else:
        print("have", f)

def load(name):
    with open(os.path.join("results", name)) as fh:
        return json.load(fh)

print("\nready — all results local.")

## 2. The headline — the 5-bar benchmark

Distilled **Qwen2.5-1.5B + LoRA** vs **bear-1.1** vs full context, QA-F1, 50 instances per
benchmark. Both get the same ratio-0.3 compression *instruction*, but ours realizes far fewer
tokens (on HotpotQA: ~48 vs bear's ~409) — so this is *more accurate at ~8.5× fewer tokens*,
not a matched-output tie.

In [ ]:
BENCH = [("hotpotqa", "HotpotQA", "in-domain"),
         ("2wiki", "2WikiMultiHop", "near-in-dist"),
         ("musique", "MuSiQue", "OOD"),
         ("squad", "SQuAD v2", "OOD single-hop")]

print(f"{'benchmark':16} {'none':>6} {'bear':>6} {'ours':>6} {'Δ vs bear':>9} {'95% CI':>18}  sig")
print("-" * 76)
rows = []
for key, label, tag in BENCH:
    d = load(f"5bar_distilled_{key}.json")
    b, dv = d["bars"], d["deltas_vs_bear"]["ours"]
    none, bear, ours = b["none"]["mean_f1"], b["bear"]["mean_f1"], b["ours"]["mean_f1"]
    delta, lo, hi, sig = dv["mean_delta"], dv["ci_lo"], dv["ci_hi"], dv["excludes_zero"]
    rows.append((label, none, bear, ours, delta, lo, hi, sig))
    print(f"{label:16} {none:6.3f} {bear:6.3f} {ours:6.3f} {delta:+9.3f} "
          f"({lo:+.3f},{hi:+.3f})  {'✓' if sig else 'n.s.'}")
print("\n✓ = 95% bootstrap CI on the paired delta excludes zero. Significant on 2 of 4.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = [r[0] for r in rows]
x = np.arange(len(labels)); w = 0.26
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - w, [r[1] for r in rows], w, label="full ctx", color="#9aa3b2")
ax.bar(x,     [r[2] for r in rows], w, label="bear-1.1", color="#ff5d5d")
ax.bar(x + w, [r[3] for r in rows], w, label="ReCompress (1.5B)", color="#2a6f97")
for i, r in enumerate(rows):
    ax.text(i + w, r[3] + 0.02, f"{r[3]:.2f}", ha="center", fontsize=9, color="#2a6f97")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("QA-F1"); ax.set_ylim(0, 1)
ax.set_title("ReCompress vs bear-1.1 across benchmarks (ratio 0.3, n=50)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 3. The cross-solver audit — does the win survive an independent judge?

Our teacher and our solver are both DeepSeek. A reviewer's first attack: maybe "ours" enjoys
solver-affinity that bear doesn't. So we re-scored HotpotQA with **Claude Sonnet** — a model
independent of *both* the teacher and the student.

In [ ]:
cs = load("cross_solver_audit.json")["cross_solver"]
for name, key in [("DeepSeek (in-family)", "deepseek"), ("Claude Sonnet (independent)", "claude_sonnet")]:
    s = cs[key]
    print(f"{name:30}  ours={s['ours_mean_f1']:.3f}  bear={s['bear_mean_f1']:.3f}  "
          f"Δ={s['delta']:+.3f}  CI=({s['ci_lo']:+.3f},{s['ci_hi']:+.3f})  "
          f"{'✓' if s['excludes_zero'] else 'n.s.'}")
print("\nThe gap is essentially identical under an independent judge — not a same-family artifact.")

## 4. Honesty: mask-the-answer — compression or extraction?

How much of the F1 is carried by the *literal answer span* being present, vs. real reasoning?
We redact the gold answer from each compression and re-solve — for **both** systems.

In [ ]:
m = load("mask_symmetric.json")
for sysname in ("ours", "bear"):
    s = m[sysname]
    print(f"{s['system']:24}  unmasked={s['unmasked_f1']:.3f}  masked={s['masked_f1']:.3f}  "
          f"drop={s['drop_pct_of_unmasked']*100:.0f}%")
print("\nMasking hits OURS harder (-65%) than bear (-31%): a large share of our margin is")
print("*span selection* (keeping the answer-bearing span at a 3.5% budget where deletion")
print("truncates it), not 'better reasoning'. We report this rather than hide it.")

## 5. Multi-turn crossover — and the Echidna self-correction

Total tokens (context sent to the solver **plus** per-turn compression overhead) as a
conversation grows. We found our LLM checkpoint-trigger ("Echidna") decided *checkpoint* 98% of
the time — useless — and replaced it with a free rule.

In [ ]:
sw = load("echidna_ablation_sweep.json")["by_turns"]
T = sorted(int(k) for k in sw)
mock = [sw[str(t)]["rezero_mock"]["total"] for t in T]
llm  = [sw[str(t)]["rezero_llm"]["total"] for t in T]
nu   = [sw[str(t)]["naive_cum_uncached"] for t in T]
nc   = [sw[str(t)]["naive_ctx_cached"] for t in T]

print(f"{'turns':>5} {'rule-Echidna':>13} {'LLM-Echidna':>12} {'naive(uncached)':>16} {'rule vs uncached':>18}")
for i, t in enumerate(T):
    print(f"{t:>5} {mock[i]:>13,} {llm[i]:>12,} {nu[i]:>16,} {nu[i]/mock[i]:>16.1f}×")

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(T, nu, "o-", color="#ff5d5d", label="naive (uncached)")
ax.plot(T, nc, "s--", color="#e8896b", label="naive (KV-cached)")
ax.plot(T, llm, "^-", color="#8b93a7", label="RbD + LLM Echidna")
ax.plot(T, mock, "D-", color="#2a6f97", lw=2.5, label="RbD + rule Echidna")
ax.set_xlabel("conversation length (turns)"); ax.set_ylabel("total tokens")
ax.set_title("Total-token crossover (ctx + overhead)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Rule-Echidna beats uncached naive from ~6 turns; ~4× cheaper by 20. Never beats a")
print("KV-cached agent on raw tokens (honest caveat).")

## 6. Read the real compressions

Browse the 50 actual HotpotQA compressions our 1.5B produced — the compressed text, token
count, and how each solver scored it. Change `i` to inspect any instance.

In [ ]:
pi = load("cross_solver_audit.json")["per_instance"]

def show(i):
    p = pi[i]
    print(f"[{i}] Q: {p['question']}")
    print(f"    gold: {p['gold']!r}")
    print(f"    ours ({p['ours_tok']} tok): {p['ours_compressed']}")
    print(f"    F1  -> DeepSeek {p['ours_f1_deepseek']:.2f} | Claude {p['ours_f1_claude']:.2f} | bear {p['bear_f1_deepseek']:.2f}")
    print(f"    gold appears verbatim in compression: {p['gold_leaked_in_ours']}")

show(0)
print("\n— change show(i) for any i in 0..49 —")
# leakage summary
leaked = sum(1 for p in pi if p["gold_leaked_in_ours"])
print(f"\nanswer-leakage: gold verbatim in {leaked}/{len(pi)} = {leaked/len(pi):.0%} of compressions")

## 7. (Optional) Run the live query-aware compressor on your own text

Parts 1–6 needed nothing. This part is **optional** and runs the *teacher* compressor (DeepSeek)
so you can watch query-aware rewriting on your own input. It needs a **DeepSeek API key**
([platform.deepseek.com](https://platform.deepseek.com)).

> The shipped *student* is a 1.5B LoRA that runs on a GPU via Modal (see Part 8). This cell uses
> the API teacher — the upper bound the student is distilled from — so it works in plain Colab.
> Leave the key blank to skip.

In [ ]:
import getpass
DEEPSEEK_API_KEY = ""  # or paste here; leave blank to be prompted / skip
if not DEEPSEEK_API_KEY:
    try:
        DEEPSEEK_API_KEY = getpass.getpass("DeepSeek API key (blank to skip): ").strip()
    except Exception:
        DEEPSEEK_API_KEY = ""

if DEEPSEEK_API_KEY:
    !pip -q install openai >/dev/null
    print("key set — run the next cell.")
else:
    print("no key — skipping the live cell. Parts 1–6 already reproduced the paper.")

In [ ]:
# Edit these, then run. Mirrors the paper's query-aware compression prompt.
CONTEXT = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
    "It is named after the engineer Gustave Eiffel, whose company designed and built the tower. "
    "Locally nicknamed 'La dame de fer', it was constructed from 1887 to 1889. "
    "The Louvre is the world's most-visited museum, located on the Right Bank of the Seine. "
    "Mount Everest is Earth's highest mountain above sea level, located in the Himalayas."
)
QUESTION = "Who is the Eiffel Tower named after?"
RATIO = 0.3

if DEEPSEEK_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
    target = max(8, int(len(CONTEXT.split()) * RATIO))
    system = (
        "You are a query-aware context compressor. Given a context and a question, output a "
        "compressed version that keeps ONLY what is needed to answer the question: drop "
        "irrelevant passages entirely and densify the rest. Do NOT answer the question; output "
        "compressed context only, no preamble."
    )
    prompt = f"Question: {QUESTION}\n\nContext:\n{CONTEXT}\n\nCompress to about {target} words."
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        max_tokens=512, temperature=0.0,
    )
    out = resp.choices[0].message.content.strip()
    print(f"ORIGINAL  ({len(CONTEXT.split())} words):\n{CONTEXT}\n")
    print(f"COMPRESSED ({len(out.split())} words, query-aware):\n{out}\n")
    print(f"→ dropped the Louvre / Everest distractors and kept the Eiffel–Gustave Eiffel link.")
else:
    print("(no key set — skipped)")

## 8. The full pipeline (Modal + GPU)

The shipped artifact is a **distilled Qwen2.5-1.5B + LoRA** student, trained on a Modal H100
for ~$10 total. That needs a GPU and a Modal account, so it isn't run here — but it's fully
reproducible from the repo:

```bash
git clone https://github.com/Kart-ing/ReCompress && cd ReCompress
pip install -r requirements.txt          # + set DEEPSEEK_API_KEY, BEAR/TTC key in .env

# 1. generate teacher data  -> data/distill/train_v3.jsonl
python -m recompress.distill.gen_data --n 5000
# 2. LoRA fine-tune on a Modal H100 (~12-15 min)
modal run --detach recompress/distill/train.py
# 3. the headline 5-bar, all benchmarks  -> results/5bar_distilled_*.json
modal run recompress/distill/eval_5bar.py
# 4. the audits this notebook reads
modal run recompress/distill/cross_solver_eval.py --n 50
python -m recompress.distill.mask_answer_symmetric
cd rezero && modal run experiments/echidna_ablation_sweep.py --turns 6,10,15,20
```

Full instructions and the failure→success trajectory are in the repo README and
`docs/MULTITURN_OVERHEAD.md`.

---

**Links** — 📄 [Paper (Zenodo)](https://doi.org/10.5281/zenodo.20786357) · 🎛 [Interactive demo](https://demo-eight-olive-97.vercel.app) · 💻 [GitHub](https://github.com/Kart-ing/ReCompress)

*ReCompress · Parth Sanjay Kshirsagar & Kartikey Pandey · The Token Company Compression Challenge · UC Berkeley AI Hackathon 2026.*